# Profiling with Nsight (Phase 10)

Runs `profiling/*.sh` on a Colab GPU and downloads the reports. Read `docs/10_nsight_profiling.md` first: it explains every number these reports contain.

Use the same GPU type as the benchmark run (T4) so profiles and benchmarks describe the same hardware.

**Upload the CURRENT project:** zip your local `cuda-transformer-performance` folder again (it must contain `profiling/ncu_kernels.sh` and `src/benchmark/profile_targets.cu`). An older zip from the benchmark run does not have the profiling files.

In [ ]:
# 1. Upload the zip of the current project folder (skip this cell if you cloned the repository instead).
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "{zip_name}" -d /content/

In [ ]:
# 2. Find the project wherever it was unzipped (it may be nested, e.g. /content/<folder>/cuda-transformer-performance),
#    using the most recently extracted copy, then build.
import glob, os
roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/**/profiling/ncu_kernels.sh', recursive=True)]
if not roots:
    raise SystemExit('No project with profiling/ncu_kernels.sh under /content. The uploaded copy is older than '
                     'Phase 10: re-zip your current cuda-transformer-performance folder and upload it again.')
root = max(roots, key=lambda r: os.path.getmtime(os.path.join(r, 'profiling', 'ncu_kernels.sh')))
os.chdir(root)
print('Project:', root)
!sed -i 's/\r$//' scripts/*.sh profiling/*.sh
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j 2>&1 | tail -n 3
!ls build/profile_targets && nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# 3. Are the profilers installed? (They ship with the CUDA toolkit; Colab usually has them.)
!which ncu nsys || ls /usr/local/cuda/bin | grep -E 'ncu|nsys' || echo 'profiler(s) missing: see docs/10 section 2'
!ncu --version | tail -n 1; nsys --version

In [ ]:
# 4. Registers / shared memory / spills per kernel (no profiler needed)
!bash profiling/resource_usage.sh

In [ ]:
# 5. Nsight Systems timeline (all cases, 5 repetitions each)
!bash profiling/nsys_timeline.sh all

In [ ]:
# 6. Nsight Compute, one case at a time (each takes about 1-3 minutes)
for case in ["gemm", "precision", "softmax", "layernorm", "attention", "vector_add"]:
    print(f"===== {case} =====")
    !bash profiling/ncu_kernels.sh {case}

In [ ]:
# 7. Download every report (text, CSV and the binary reports for the Nsight GUIs)
!zip -q -r profiling_reports.zip profiling/reports
from google.colab import files
files.download("profiling_reports.zip")